<a href="https://colab.research.google.com/github/ababacaryoro/nlp-course/blob/main/assignments/1/assignment1_part1_regex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 – Part 1: regular expressions and date extraction

**Points:** 10 (half of Assignment 1)

**Rules**
1. Complete every function marked `# YOUR CODE HERE`.
2. Do **not** change the function names or their signatures, and return exactly the type stated.
3. Test each function with the cell that follows it.
4. Work alone. Using an AI assistant to write the answers is not allowed.

## The data

`data/dates.txt` contains 500 lines of medical notes. Each line contains **one date**, written in one of many formats:

- `04/20/2009; 04/20/09; 4/20/09; 4/3/09`
- `Mar-20-2009; Mar 20, 2009; March 20, 2009; Mar. 20, 2009; Mar 20 2009`
- `20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009`
- `Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009`
- `Feb 2009; Sep 2009; Oct 2010`
- `6/2008; 12/2009`
- `2009; 2010`

## Setup

In [11]:
# Google Colab only: install the libraries that Colab does not ship (about 1 minute).
# Uncomment the next line and run the cell. Nothing to do for a local installation made with `uv sync`.
# !pip install -q -r https://raw.githubusercontent.com/ababacaryoro/nlp-course/main/requirements.txt

In [12]:
import os, re, urllib.request
import pandas as pd
import numpy as np
from datetime import datetime

def local_file(name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/ababacaryoro/nlp-course/main/assignments/1/data/{name}", path)
    return path

with open(local_file("dates.txt"), encoding="utf-8") as f:
    df = pd.Series([line for line in f])
print(f"{len(df)} notes")
df.head()

500 notes


0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
dtype: object

## Question 1 (1 point)

Return the list of every date written as `MM/DD/YY` or `MM/DD/YYYY` (for example `03/25/93`, `6/18/85`, `5/24/1990`) found in the notes.

In [13]:
def question_one():
    """List of all MM/DD/YY or MM/DD/YYYY strings found in the notes."""
    # Also accept the one extra leading zero present in the supplied data (e.g. 011/14/83).
    pattern = r"(?<!\d)0?(?:[1-9]|1[0-2])/(?:0?[1-9]|[12]\d|3[01])/(?:\d{2}|\d{4})(?!\d)"
    results = []
    for note in df:
        results.extend(re.findall(pattern, note))
    return results

q1 = question_one()
print(f"{len(q1)} dates, first 10: {q1[:10]}")


121 dates, first 10: ['03/25/93', '6/18/85', '7/8/71', '9/27/75', '2/6/96', '7/06/79', '5/18/78', '10/24/89', '3/7/86', '4/10/71']


## Question 2 (1 point)

Return the list of every date that starts with a month name followed by a day and a year: `Mar-20-2009`, `Mar 20, 2009`, `March 20, 2009`, `Mar. 20, 2009`, `Mar 20 2009`, `Mar 20th, 2009`.

In [14]:
def question_two():
    """List of all 'Month day, year' strings (full or abbreviated month name) found in the notes."""
    month_pattern = r"(?:January|Jan|February|Feb|March|Mar|April|Apr|May|June|Jun|July|Jul|August|Aug|September|Sept|Sep|October|Oct|November|Nov|December|Dec)"
    pattern = rf"\b{month_pattern}\b\.?(?:\s+|-)\d{{1,2}}(?:st|nd|rd|th)?(?:,\s*|\s+|-)\d{{4}}\b"
    results = []
    for note in df:
        results.extend(re.findall(pattern, note, flags=re.IGNORECASE))
    return results

q2 = question_two()
print(f"{len(q2)} dates, first 10: {q2[:10]}")


34 dates, first 10: ['April 11, 1990', 'May 30, 2001', 'Feb 18, 1994', 'February 18, 1981', 'October. 11, 2013', 'Jan 24 1986', 'July 26, 1978', 'December 23, 1999', 'May 15, 1989', 'September 06, 1995']


## Question 3 (1 point)

Return the list of every date written as `DD Month YYYY`: `20 Mar 2009`, `20 March 2009`, `20 Mar. 2009`, `20 March, 2009`.

In [15]:
def question_three():
    """List of all 'day Month year' strings found in the notes."""
    month_pattern = r"(?:January|Jan|February|Feb|March|Mar|April|Apr|May|June|Jun|July|Jul|August|Aug|September|Sept|Sep|October|Oct|November|Nov|December|Dec)"
    pattern = rf"\b\d{{1,2}}\s+{month_pattern}\b\.?[,]?\s+\d{{4}}\b"
    results = []
    for note in df:
        results.extend(re.findall(pattern, note, flags=re.IGNORECASE))
    return results

q3 = question_three()
print(f"{len(q3)} dates, first 10: {q3[:10]}")


69 dates, first 10: ['24 Jan 2001', '10 Sep 2004', '26 May 1982', '28 June 2002', '06 May 1972', '25 Oct 1987', '14 Oct 1996', '30 Nov 2007', '28 June 1994', '14 Jan 1981']


## Question 4 (1 point)

`question_four(text)` returns the list of e-mail addresses found in `text`.

In [16]:
def question_four(text):
    """List of the e-mail addresses in text."""
    pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    return re.findall(pattern, text)

test_text = """
Contact us at support@company.com or sales@company.org.
You can also reach john.doe@email.co.uk or jane_doe123@university.edu.
Invalid: @invalid.com, user@, not-an-email
"""
print(question_four(test_text))
# Expected: ['support@company.com', 'sales@company.org', 'john.doe@email.co.uk', 'jane_doe123@university.edu']


['support@company.com', 'sales@company.org', 'john.doe@email.co.uk', 'jane_doe123@university.edu']


## Question 5 (1 point)

`question_five(text)` cleans a text: remove digits, remove punctuation, lowercase, normalize whitespace.

In [17]:
def question_five(text):
    """Cleaned text: no digit, no punctuation, lowercase, single spaces."""
    text = text.lower()
    text = re.sub(r"\d", "", text)
    text = re.sub(r"[^\w\s]|_", "", text)
    return " ".join(text.split())

print(repr(question_five("Hello, World! 123 This is a TEST... with 456 numbers!!!")))
# Expected: 'hello world this is a test with numbers'


'hello world this is a test with numbers'


## Question 6 (2 points)

`question_six(text)` extracts the phone numbers written as `XXX-XXX-XXXX`, `(XXX) XXX-XXXX`, `XXX.XXX.XXXX` or `XXX XXX XXXX` and returns them normalized as `XXX-XXX-XXXX`. Sequences such as `12-34-5678` or `1234567890` are not valid.

In [18]:
def question_six(text):
    """List of phone numbers normalized as XXX-XXX-XXXX."""
    pattern = re.compile(
        r"(?<!\d)(?:\d{3}-\d{3}-\d{4}|\(\d{3}\)[ ]*\d{3}-\d{4}|"
        r"\d{3}\.\d{3}\.\d{4}|\d{3}[ ]+\d{3}[ ]+\d{4})(?!\d)"
    )
    results = []
    for match in pattern.finditer(text):
        digits = re.sub(r"\D", "", match.group())
        results.append(f"{digits[:3]}-{digits[3:6]}-{digits[6:]}")
    return results

test_text = """
Call us at 123-456-7890 or (555) 123-4567.
You can also reach us at 888.555.1234 or 999 888 7777.
Invalid: 12-34-5678, 1234567890
"""
print(question_six(test_text))
# Expected: ['123-456-7890', '555-123-4567', '888-555-1234', '999-888-7777']


['123-456-7890', '555-123-4567', '888-555-1234', '999-888-7777']


## Question 7 (3 points)

Extract the date of every note and sort the notes chronologically.

**Rules**
- `xx/xx/xx` is `mm/dd/yy`.
- A 2-digit year is in the 1900s (`1/5/89` is January 5th, 1989).
- When the day is missing (`9/2009`, `Feb 2009`), use the first day of the month.
- When the month is missing (`2010`), use January 1st.

`question_seven()` returns a `pandas.Series` of length 500 whose **values are the original indices** of the notes, in ascending chronological order.

Example: if the notes were `0: "1999"`, `1: "2010"`, `2: "1978"`, the function returns the Series `[2, 0, 1]`.

In [19]:
def _to_datetime(kind, fields):
    """Convert one matched date's named fields to a datetime."""
    month_numbers = {"jan": 1, "feb": 2, "mar": 3, "apr": 4, "may": 5, "jun": 6,
                     "jul": 7, "aug": 8, "sep": 9, "oct": 10, "nov": 11, "dec": 12}
    month_names = ["january", "february", "march", "april", "may", "june",
                   "july", "august", "september", "october", "november", "december"]

    def month_number(token, allow_prefix_noise=False):
        candidates = [token.lower()]
        if allow_prefix_noise and len(token) > 1:
            candidates.append(token[1:].lower())

        for candidate in candidates:
            if candidate[:3] in month_numbers and candidate in {
                "jan", "january", "feb", "february", "mar", "march", "apr", "april",
                "may", "jun", "june", "jul", "july", "aug", "august", "sep", "sept",
                "september", "oct", "october", "nov", "november", "dec", "december",
            }:
                return month_numbers[candidate[:3]]

        fuzzy_matches = set()
        for candidate in candidates:
            for number, name in enumerate(month_names, start=1):
                if abs(len(candidate) - len(name)) > 1:
                    continue
                if len(candidate) == len(name):
                    differences = sum(a != b for a, b in zip(candidate, name))
                    if differences <= 1:
                        fuzzy_matches.add(number)
                else:
                    shorter, longer = sorted((candidate, name), key=len)
                    i = j = differences = 0
                    while i < len(shorter) and j < len(longer):
                        if shorter[i] == longer[j]:
                            i += 1
                            j += 1
                        else:
                            differences += 1
                            j += 1
                            if differences > 1:
                                break
                    differences += len(longer) - j
                    if differences <= 1:
                        fuzzy_matches.add(number)
        if len(fuzzy_matches) == 1:
            return fuzzy_matches.pop()
        raise ValueError(f"Unrecognized or ambiguous month: {token!r}")

    if kind == "numeric_date":
        year = int(fields["year"])
        if len(fields["year"]) == 2:
            year += 1900
        return datetime(year, int(fields["month"]), int(fields["day"]))
    if kind in ("month_day_year", "day_month_year"):
        return datetime(int(fields["year"]), month_number(fields["month"]), int(fields["day"]))
    if kind == "numeric_month_year":
        return datetime(int(fields["year"]), int(fields["month"]), 1)
    if kind == "month_year":
        return datetime(int(fields["year"]), month_number(fields["month"], allow_prefix_noise=True), 1)
    if kind == "year":
        return datetime(int(fields["year"]), 1, 1)
    raise ValueError(f"Unknown date format: {kind}")

def question_seven():
    """Series of the 500 note indices sorted by date."""
    # Capture the month word; _to_datetime validates and normalizes it.
    month_pattern = r"[A-Za-z]+"
    date_patterns = [
        ("numeric_date", re.compile(
            r"(?<!\d)(?P<month>0?(?:[1-9]|1[0-2]))[/-](?P<day>0?[1-9]|[12]\d|3[01])[/-](?P<year>\d{2}|\d{4})(?!\d)"
        )),
        ("month_day_year", re.compile(
            rf"(?<![A-Za-z])(?P<month>{month_pattern})\b\.?(?:\s+|-)(?P<day>\d{{1,2}})(?:st|nd|rd|th)?(?:,\s*|\s+|-)(?P<year>\d{{4}})(?!\d)",
            re.IGNORECASE,
        )),
        ("day_month_year", re.compile(
            rf"(?<!\d)(?P<day>\d{{1,2}})(?:st|nd|rd|th)?\s+(?P<month>{month_pattern})\b\.?[,]?\s+(?P<year>\d{{4}})(?!\d)",
            re.IGNORECASE,
        )),
        ("numeric_month_year", re.compile(
            r"(?<!\d)(?P<month>0?[1-9]|1[0-2])/(?P<year>\d{4})(?!\d)"
        )),
        # Some notes have one stray letter attached to the month (e.g. pOct 2015).
        ("month_year", re.compile(
            rf"(?<![A-Za-z])(?P<month>{month_pattern})\b\.?[,]?\s+(?:of\s+)?(?P<year>\d{{4}})(?!\d)",
            re.IGNORECASE,
        )),
        ("year", re.compile(r"(?<!\d)(?P<year>(?:19|20)\d{2})(?!\d)")),
    ]

    dated_indices = []
    for index, note in enumerate(df):
        parsed_date = None
        for kind, pattern in date_patterns:  # Most specific format first.
            for match in pattern.finditer(str(note)):
                try:
                    parsed_date = _to_datetime(kind, match.groupdict())
                except ValueError:
                    continue
                break
            if parsed_date is not None:
                break
        if parsed_date is None:
            raise ValueError(f"Could not find a supported date in note {index}: {note!r}")
        dated_indices.append((parsed_date, index))

    dated_indices.sort(key=lambda item: (item[0], item[1]))
    return pd.Series([index for _, index in dated_indices], dtype=int)

q7 = question_seven()
print(f"Length: {len(q7)}, first 10 indices: {list(q7.head(10))}, last 10: {list(q7.tail(10))}")


Length: 500, first 10 indices: [9, 84, 2, 53, 28, 474, 153, 13, 129, 98], last 10: [152, 235, 464, 253, 231, 427, 141, 186, 161, 413]


## Check before submitting

In [20]:
checks = [("question_one", lambda: question_one(), list), ("question_two", lambda: question_two(), list),
          ("question_three", lambda: question_three(), list), ("question_four", lambda: question_four("a@b.com"), list),
          ("question_five", lambda: question_five("Hello 1"), str), ("question_six", lambda: question_six("123-456-7890"), list),
          ("question_seven", lambda: question_seven(), pd.Series)]
for name, call, typ in checks:
    try:
        r = call()
        assert isinstance(r, typ), f"should return {typ.__name__}"
        if name == "question_seven":
            assert len(r) == 500, "should have length 500"
        print(f"OK   {name}")
    except Exception as e:
        print(f"FAIL {name}: {e}")

OK   question_one
OK   question_two
OK   question_three
OK   question_four
OK   question_five
OK   question_six
OK   question_seven


---
## Submission

1. Run the whole notebook (`Kernel > Restart & Run All`): the last cell checks that every function exists and returns the right type.
2. Commit the notebook with its outputs in `assignments/1/` of your course repository and push.
3. Send an e-mail to the teacher to say that the assignment is ready (the repository link is only needed in your first e-mail).

Grading is automatic: the functions are imported and compared with reference results. Do not rename them and do not change their signatures.